# 17 · Sevilla Hidden Gems Ranking Pilot

Este notebook construye el **primer ranking piloto de Hidden Gems para Sevilla** a partir de las señales `local + plato` generadas en el notebook 16.

La idea principal no es elegir simplemente los locales con mejor valoración, sino detectar combinaciones:

> **plato concreto + local concreto + barrio/distrito**

con evidencia suficiente, sentimiento positivo, ruido controlado y una señal razonablemente diferencial.

## Entrada

- `data/artifacts/ai/sevilla/aggregation/sevilla_place_dish_ranking_candidates_v1.jsonl`
- `data/artifacts/ai/sevilla/aggregation/sevilla_global_dish_signals_v1.csv`

## Salida

- Ranking completo de candidatos.
- Ranking seleccionado `sevilla_pilot`.
- Top por barrio.
- Top por distrito.
- Top por plato.
- Resumen JSON de la ejecución.


In [1]:
# ============================================================
# 01. Imports y configuración general
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import ast
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 180)

NOTEBOOK_NAME = "17_sevilla_hidden_gems_ranking_pilot"
VERSION = "sevilla_hidden_gems_ranking_pilot_v1"
RANKING_SCOPE = "sevilla_pilot"

PROJECT_ROOT = Path.cwd()

# Si estás ejecutando el notebook desde /notebooks, subimos un nivel.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "data" / "artifacts" / "ai" / "sevilla" / "aggregation"
OUTPUT_DIR = PROJECT_ROOT / "data" / "artifacts" / "ai" / "sevilla" / "ranking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_JSONL_PATH = INPUT_DIR / "sevilla_place_dish_ranking_candidates_v1.jsonl"
CANDIDATES_CSV_PATH = INPUT_DIR / "sevilla_place_dish_ranking_candidates_v1.csv"
GLOBAL_DISH_SIGNALS_PATH = INPUT_DIR / "sevilla_global_dish_signals_v1.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CANDIDATES_JSONL_PATH:", CANDIDATES_JSONL_PATH)
print("CANDIDATES_CSV_PATH:", CANDIDATES_CSV_PATH)
print("GLOBAL_DISH_SIGNALS_PATH:", GLOBAL_DISH_SIGNALS_PATH)


PROJECT_ROOT: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline
INPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation
OUTPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking
CANDIDATES_JSONL_PATH: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation\sevilla_place_dish_ranking_candidates_v1.jsonl
CANDIDATES_CSV_PATH: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation\sevilla_place_dish_ranking_candidates_v1.csv
GLOBAL_DISH_SIGNALS_PATH: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation\sevilla_global_dish_signals_v1.csv


In [2]:
# ============================================================
# 02. Utilidades de lectura/escritura y serialización
# ============================================================

from datetime import date, datetime


def to_builtin(obj):
    """
    Convierte objetos numpy/pandas/datetime a tipos Python seguros para JSON.

    Importante:
    - Sustituye NaN, NaT, Infinity y -Infinity por None.
    - Usa allow_nan=False al guardar para garantizar JSON estricto.
    - Limpia caracteres nulos y surrogates en strings.
    """
    if obj is None:
        return None

    if isinstance(obj, dict):
        return {str(to_builtin(k)): to_builtin(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [to_builtin(v) for v in obj]

    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)

    if isinstance(obj, (np.integer, int)) and not isinstance(obj, bool):
        return int(obj)

    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        if not math.isfinite(value):
            return None
        return value

    if isinstance(obj, (pd.Timestamp, datetime, date)):
        if pd.isna(obj):
            return None
        return obj.isoformat()

    if not isinstance(obj, (list, dict, tuple, set)):
        try:
            if pd.isna(obj):
                return None
        except Exception:
            pass

    if isinstance(obj, str):
        # Evita caracteres nulos y problemas de unicode/surrogates.
        return obj.replace("\x00", "").encode("utf-8", "replace").decode("utf-8")

    return str(obj)


def safe_json_dump(obj, path: Path) -> None:
    """Guarda JSON estricto y valida que se pueda volver a leer."""
    path.parent.mkdir(parents=True, exist_ok=True)
    safe_obj = to_builtin(obj)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        json.dump(safe_obj, f, indent=2, ensure_ascii=False, allow_nan=False)

    # Validación inmediata: si esto falla, el JSON no es válido.
    with path.open("r", encoding="utf-8") as f:
        json.load(f)


def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON inválido en línea {line_no} de {path}: {exc}") from exc
    return pd.DataFrame(rows)


def save_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(to_builtin(record), ensure_ascii=False, allow_nan=False) + "\n")


def read_candidates() -> pd.DataFrame:
    if CANDIDATES_JSONL_PATH.exists():
        print("Leyendo candidatos desde JSONL:", CANDIDATES_JSONL_PATH)
        return read_jsonl(CANDIDATES_JSONL_PATH)
    if CANDIDATES_CSV_PATH.exists():
        print("Leyendo candidatos desde CSV:", CANDIDATES_CSV_PATH)
        return pd.read_csv(CANDIDATES_CSV_PATH)
    raise FileNotFoundError(
        "No se encontró ni sevilla_place_dish_ranking_candidates_v1.jsonl "
        "ni sevilla_place_dish_ranking_candidates_v1.csv. Ejecuta antes el notebook 16."
    )


def parse_maybe_list(value):
    """Convierte strings tipo lista en listas reales, manteniendo listas ya existentes."""
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and not math.isfinite(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                return parsed if isinstance(parsed, list) else [str(parsed)]
            except Exception:
                return [text]
        return [text]
    return [str(value)]


def coerce_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "1": True, "yes": True, "y": True, "false": False, "0": False, "no": False, "n": False})
        .fillna(False)
        .astype(bool)
    )


def normalize_0_1(series: pd.Series, lower=None, upper=None) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if lower is None:
        lower = float(s.min()) if len(s) else 0.0
    if upper is None:
        upper = float(s.max()) if len(s) else 1.0
    if math.isclose(upper, lower):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return ((s - lower) / (upper - lower)).clip(0.0, 1.0)


def safe_div(num, den, default=0.0):
    try:
        if den == 0 or pd.isna(den):
            return default
        result = num / den
        return result if math.isfinite(float(result)) else default
    except Exception:
        return default


def stable_id(*parts: str, prefix: str = "rank_sevilla_v1") -> str:
    raw = "|".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:18]
    return f"{prefix}_{digest}"


In [3]:
# ============================================================
# 03. Carga y validación de entradas
# ============================================================

candidates_df = read_candidates()

if not GLOBAL_DISH_SIGNALS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {GLOBAL_DISH_SIGNALS_PATH}. Ejecuta antes el notebook 16."
    )

global_dish_df = pd.read_csv(GLOBAL_DISH_SIGNALS_PATH)

print("Candidates shape:", candidates_df.shape)
print("Global dish signals shape:", global_dish_df.shape)

display(candidates_df.head(5))
display(global_dish_df.head(5))

REQUIRED_CANDIDATE_COLUMNS = [
    "place_id", "place_name",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "dish_id_v1", "display_dish_name_es_v1", "normalized_dish_name_es_v1",
    "dish_family_es_v1", "dish_group_es_v1",
    "mention_count", "review_count",
    "positive_count", "neutral_count", "negative_count",
    "positive_ratio", "negative_ratio",
    "smoothed_positive_ratio", "smoothed_negative_ratio",
    "weighted_sentiment_score",
    "avg_sentiment_confidence", "local_context_ratio", "fallback_ratio",
    "total_signal_weight", "avg_rating",
    "evidence_tier_v1", "signal_quality_tier_v1",
    "preliminary_signal_score_v1", "ranking_ready_v1",
]

missing = [c for c in REQUIRED_CANDIDATE_COLUMNS if c not in candidates_df.columns]
if missing:
    raise KeyError(f"Faltan columnas obligatorias en candidatos: {missing}")

REQUIRED_GLOBAL_COLUMNS = [
    "dish_id_v1",
    "global_mention_count",
    "global_review_count",
    "global_place_count",
    "global_neighborhood_count",
    "global_district_count",
    "global_positive_ratio",
    "global_negative_ratio",
    "global_weighted_sentiment_score",
    "global_avg_confidence",
    "global_avg_rating",
    "global_popularity_score_v1",
    "global_dish_signal_score_v1",
]

missing_global = [c for c in REQUIRED_GLOBAL_COLUMNS if c not in global_dish_df.columns]
if missing_global:
    raise KeyError(f"Faltan columnas obligatorias en señales globales: {missing_global}")

basic_checks = {
    "has_candidates": bool(len(candidates_df) > 0),
    "has_global_dish_signals": bool(len(global_dish_df) > 0),
    "candidate_place_ids_present": bool(candidates_df["place_id"].notna().all()),
    "candidate_dish_ids_present": bool(candidates_df["dish_id_v1"].notna().all()),
    "candidate_neighborhoods_present": bool(candidates_df["neighborhood_id"].notna().all()),
    "candidate_districts_present": bool(candidates_df["district_id"].notna().all()),
    "global_dish_ids_are_unique": bool(global_dish_df["dish_id_v1"].nunique(dropna=True) == len(global_dish_df)),
}

print(json.dumps(basic_checks, indent=2, ensure_ascii=False))


Leyendo candidatos desde JSONL: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation\sevilla_place_dish_ranking_candidates_v1.jsonl
Candidates shape: (256, 53)
Global dish signals shape: (181, 20)


,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,evidence_tier_v1,signal_quality_tier_v1,preliminary_signal_score_v1,ranking_ready_v1,candidate_tier_v1
0,52d0c5c7-da1f-4733-a8a1-6d3fcac2d18d,restaurante asiático shan,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,7040f0c8-e891-4ff2-ac79-044be0f73fff,BELLAVISTA,dish_es_v1_451dcf9913bf3b32,sushi,sushi,internacional,internacional,3,3,3,3,0,0,1.0000,0.0000,0.0,2.8500,0.0000,0.0,2.8500,0.6581,0.1709,1.0000,1.0000,4.9497,4.9497,0.9500,0.9500,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],5.0000,5.0,5.0,"Comida riquísima, el sushi flambeado y el yakisoba muy bueno.",,,solid,usable_signal,62.9487,True,weak_candidate
1,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,ae182ebc-43a4-46db-9364-d006cf3a73c9,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,9,5,5,7,2,0,0.7778,0.2222,0.0,4.8419,0.1679,0.0,5.0098,0.7293,0.1248,0.7778,0.9665,1.6851,2.4826,0.6578,0.5566,5,4,0.5556,0.4444,4,0,0,1,0,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, intensifier_nearby, neutral_or_ambiguous_terms_detected, contrast_...",4.2222,3.0,5.0,"Como platos principales pedimos pizzas, y fue todo un acierto: una fuera de carta con guiso de carne, increíblemente sabrosa y diferente, y otra de carta, ahumada, con un sabor...",,"Llegamos con ganas de comer una pizza y una bola de arroz rellena de carne y salsa, para nuestra sorpresa nos sentamos en mesa y nos dicen que solo hacen pizza, igual habrá que...",strong,high_quality_signal,62.1079,True,not_ready
2,a3db8f79-3216-4c52-8307-7730e9335737,Tarannà,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,903feb9b-aed1-4717-abbf-e693ac5585a0,SANTA CATALINA,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,3,3,3,3,0,0,1.0000,0.0000,0.0,2.6075,0.0000,0.0,2.6075,0.6433,0.1783,1.0000,1.0000,3.1247,3.3604,0.8692,0.8692,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],4.6667,4.0,5.0,Tomamos un tartar de atún de almadraba súper recomendable y un arroz súper jugoso.,,,solid,usable_signal,60.1190,True,weak_candidate
3,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,f36f071a-eedf-4022-adcb-a0d67a0ccf28,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,6,5,5,6,0,0,1.0000,0.0000,0.0,4.4098,0.0000,0.0,4.4098,0.7301,0.1350,1.0000,1.0000,4.6146,5.3569,0.8350,0.7350,5,1,0.8333,0.1667,1,1,1,0,0,"[intensifier_nearby, low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, price_context_detected, sentiment_may_refer_to...",5.0000,5.0,5.0,"Unas experiencia muy buena la pizza 10 de 10 me encanto, esta vez fue la carnivora y seguiré probando las demás pizza q tienen, 100 % recomendados",,,strong,high_quality_signal,60.0726,True,not_ready
4,1844aa47-4f98-47f5-a31d-0f5565cd4990,ALCÁZAR ANDALUSÍ TAPAS,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,dd0845a6-120f-4945-9a3d-ec166c36337c,FERI

,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,global_mention_count,global_review_count,global_place_count,global_neighborhood_count,global_district_count,global_positive_count,global_neutral_count,global_negative_count,global_positive_ratio,global_negative_ratio,global_weighted_sentiment_score,global_avg_confidence,global_avg_rating,global_popularity_score_v1,global_dish_signal_score_v1
0,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,ensaladilla,216.0,197.0,153.0,57.0,11.0,167.0,18.0,31.0,0.7731,0.1435,1.8679,0.6275,4.1806,1.0000,83.8813
1,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,tapas_clasicas,croqueta,211.0,198.0,160.0,61.0,11.0,166.0,17.0,28.0,0.7867,0.1327,1.6126,0.6072,4.3128,0.9957,82.8871
2,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,carrillada,172.0,152.0,123.0,62.0,11.0,149.0,12.0,11.0,0.8663,0.0640,2.2245,0.6390,4.4884,0.9579,87.6920
3,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,tapas_clasicas,solomillo,167.0,152.0,126.0,57.0,11.0,132.0,10.0,25.0,0.7904,0.1497,1.9356,0.6495,4.2096,0.9524,84.8121
4,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,117.0,99.0,70.0,38.0,10.0,94.0,8.0,15.0,0.8034,0.1282,1.8923,0.7066,4.4017,0.8868,85.5311


{
  "has_candidates": true,
  "has_global_dish_signals": true,
  "candidate_place_ids_present": true,
  "candidate_dish_ids_present": true,
  "candidate_neighborhoods_present": true,
  "candidate_districts_present": true,
  "global_dish_ids_are_unique": true
}


In [4]:
# ============================================================
# 04. Preparación de datos y unión con señales globales
# ============================================================

ranking_df = candidates_df.copy()

# Booleanos y listas
ranking_df["ranking_ready_v1"] = coerce_bool_series(ranking_df["ranking_ready_v1"])
ranking_df["top_sentiment_flags_v1"] = ranking_df.get("top_sentiment_flags_v1", pd.Series([[]] * len(ranking_df))).apply(parse_maybe_list)

# Aseguramos numéricos
NUMERIC_COLUMNS = [
    "mention_count", "review_count", "source_review_count",
    "positive_count", "neutral_count", "negative_count",
    "positive_ratio", "neutral_ratio", "negative_ratio",
    "weighted_positive_count", "weighted_neutral_count", "weighted_negative_count",
    "total_signal_weight", "smoothed_positive_ratio", "smoothed_negative_ratio",
    "avg_sentiment_value", "weighted_sentiment_value",
    "avg_sentiment_score", "weighted_sentiment_score",
    "avg_sentiment_confidence", "avg_aggregation_weight",
    "local_context_count", "review_prior_fallback_count",
    "local_context_ratio", "fallback_ratio",
    "low_confidence_flag_count", "service_flag_count", "price_flag_count",
    "contrast_flag_count", "negation_flag_count",
    "avg_rating", "min_rating", "max_rating",
    "preliminary_signal_score_v1",
]

for col in NUMERIC_COLUMNS:
    if col in ranking_df.columns:
        ranking_df[col] = pd.to_numeric(ranking_df[col], errors="coerce").fillna(0.0)

global_cols_to_merge = [
    "dish_id_v1",
    "global_mention_count",
    "global_review_count",
    "global_place_count",
    "global_neighborhood_count",
    "global_district_count",
    "global_positive_ratio",
    "global_negative_ratio",
    "global_weighted_sentiment_score",
    "global_avg_confidence",
    "global_avg_rating",
    "global_popularity_score_v1",
    "global_dish_signal_score_v1",
]

ranking_df = ranking_df.merge(
    global_dish_df[global_cols_to_merge],
    on="dish_id_v1",
    how="left",
)

# Rellenos seguros
for col in global_cols_to_merge:
    if col != "dish_id_v1" and col in ranking_df.columns:
        ranking_df[col] = pd.to_numeric(ranking_df[col], errors="coerce").fillna(0.0)

ranking_df["candidate_pair_id_v1"] = ranking_df.apply(
    lambda r: stable_id(r.get("place_id"), r.get("dish_id_v1"), prefix="place_dish_candidate_v1"),
    axis=1,
)

print("Ranking dataframe:", ranking_df.shape)
display(ranking_df.head(5))


Ranking dataframe: (256, 66)


,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,evidence_tier_v1,signal_quality_tier_v1,preliminary_signal_score_v1,ranking_ready_v1,candidate_tier_v1,global_mention_count,global_review_count,global_place_count,global_neighborhood_count,global_district_count,global_positive_ratio,global_negative_ratio,global_weighted_sentiment_score,global_avg_confidence,global_avg_rating,global_popularity_score_v1,global_dish_signal_score_v1,candidate_pair_id_v1
0,52d0c5c7-da1f-4733-a8a1-6d3fcac2d18d,restaurante asiático shan,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,7040f0c8-e891-4ff2-ac79-044be0f73fff,BELLAVISTA,dish_es_v1_451dcf9913bf3b32,sushi,sushi,internacional,internacional,3,3,3,3,0,0,1.0000,0.0000,0.0,2.8500,0.0000,0.0,2.8500,0.6581,0.1709,1.0000,1.0000,4.9497,4.9497,0.9500,0.9500,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],5.0000,5.0,5.0,"Comida riquísima, el sushi flambeado y el yakisoba muy bueno.",,,solid,usable_signal,62.9487,True,weak_candidate,16.0,14.0,7.0,7.0,6.0,0.8750,0.0000,3.5798,0.7273,4.7500,0.5266,82.5426,place_dish_candidate_v1_1224ba92aabf4e0bb5
1,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,ae182ebc-43a4-46db-9364-d006cf3a73c9,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,9,5,5,7,2,0,0.7778,0.2222,0.0,4.8419,0.1679,0.0,5.0098,0.7293,0.1248,0.7778,0.9665,1.6851,2.4826,0.6578,0.5566,5,4,0.5556,0.4444,4,0,0,1,0,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, intensifier_nearby, neutral_or_ambiguous_terms_detected, contrast_...",4.2222,3.0,5.0,"Como platos principales pedimos pizzas, y fue todo un acierto: una fuera de carta con guiso de carne, increíblemente sabrosa y diferente, y otra de carta, ahumada, con un sabor...",,"Llegamos con ganas de comer una pizza y una bola de arroz rellena de carne y salsa, para nuestra sorpresa nos sentamos en mesa y nos dicen que solo hacen pizza, igual habrá que...",strong,high_quality_signal,62.1079,True,not_ready,66.0,50.0,20.0,18.0,7.0,0.8030,0.0758,2.4492,0.6421,4.1818,0.7816,86.5497,place_dish_candidate_v1_69a6a304185711c69d
2,a3db8f79-3216-4c52-8307-7730e9335737,Tarannà,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,903feb9b-aed1-4717-abbf-e693ac5585a0,SANTA CATALINA,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,3,3,3,3,0,0,1.0000,0.0000,0.0,2.6075,0.0000,0.0,2.6075,0.6433,0.1783,1.0000,1.0000,3.1247,3.3604,0.8692,0.8692,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],4.6667,4.0,5.0,Tomamos un tartar de atún de almadraba súper recomendable y un arroz súper jugoso.,,,solid,usable_signal,60.1190,True,weak_candidate,117.0,99.0,70.0,38.0,10.0,0.8034,0.1282,1.8923,0.7066,4.4017,0.8868,85.5311,place_dish_candidate_v1_f20cf8183dc7068658
3,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,f36f071a-eedf-4022-adcb-a0d67a0ccf28,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,6,5,5,6,0,0,1.0000,0.0000,0.0,4.4098,0.0

## 05. Fórmula de ranking

El ranking combina cinco dimensiones:

1. **Calidad local de la señal**: score previo, sentimiento suavizado, ratio positivo/negativo, rating medio.
2. **Evidencia**: número de menciones, número de reviews, peso total y tier de evidencia.
3. **Confianza**: confianza media, porcentaje de sentimiento local frente a fallback y ausencia de señales débiles.
4. **Hiddenness / diferenciación**: penaliza platos demasiado extendidos globalmente y premia señales menos comunes en el barrio.
5. **Penalizaciones**: ruido, fallback, precio/servicio y ratio negativo.

El resultado sigue siendo una versión piloto. No debe interpretarse como ranking productivo final hasta cargarlo, auditarlo y validarlo con muestras manuales.


In [5]:
# ============================================================
# 05. Componentes de scoring
# ============================================================

def clip01(series_or_value):
    return np.clip(series_or_value, 0.0, 1.0)


evidence_tier_score = {
    "weak": 0.15,
    "emerging": 0.45,
    "solid": 0.75,
    "strong": 1.00,
}

signal_quality_score = {
    "single_mention_signal": 0.15,
    "weak_signal": 0.30,
    "noisy_signal": 0.25,
    "emerging_signal": 0.50,
    "usable_signal": 0.75,
    "high_quality_signal": 1.00,
}

# Calidad local
ranking_df["quality_component_v1"] = clip01(
    (ranking_df["preliminary_signal_score_v1"] / 100.0) * 0.30
    + ranking_df["smoothed_positive_ratio"].clip(0, 1) * 0.25
    + (1.0 - ranking_df["smoothed_negative_ratio"].clip(0, 1)) * 0.15
    + ((ranking_df["weighted_sentiment_score"] + 2.0) / 8.0).clip(0, 1) * 0.20
    + (ranking_df["avg_rating"] / 5.0).clip(0, 1) * 0.10
)

# Evidencia
ranking_df["evidence_component_v1"] = clip01(
    (np.log1p(ranking_df["mention_count"]) / np.log1p(10)).clip(0, 1) * 0.35
    + (np.log1p(ranking_df["review_count"]) / np.log1p(5)).clip(0, 1) * 0.35
    + (ranking_df["total_signal_weight"] / 6.0).clip(0, 1) * 0.20
    + ranking_df["evidence_tier_v1"].map(evidence_tier_score).fillna(0.20) * 0.10
)

# Confianza
ranking_df["confidence_component_v1"] = clip01(
    ranking_df["avg_sentiment_confidence"].clip(0, 1) * 0.45
    + ranking_df["local_context_ratio"].clip(0, 1) * 0.30
    + (1.0 - ranking_df["fallback_ratio"].clip(0, 1)) * 0.20
    + (1.0 - (ranking_df["low_confidence_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1)) * 0.05
)

# Diferenciación en barrio: si muchos locales del mismo barrio mencionan el mismo plato, baja ligeramente.
neighborhood_dish_place_count = ranking_df.groupby(
    ["neighborhood_id", "dish_id_v1"],
    dropna=False
)["place_id"].transform("nunique")

ranking_df["neighborhood_dish_place_count_v1"] = neighborhood_dish_place_count

max_neighborhood_dish_places = max(1, int(neighborhood_dish_place_count.max()))
ranking_df["neighborhood_uniqueness_component_v1"] = (
    1.0 - (np.log1p(neighborhood_dish_place_count - 1) / np.log1p(max_neighborhood_dish_places))
).clip(0, 1)

# Hiddenness global: los platos muy extendidos en muchos locales se penalizan solo de forma moderada.
max_global_places = max(1, int(ranking_df["global_place_count"].max()))
global_commonness = (np.log1p(ranking_df["global_place_count"]) / np.log1p(max_global_places)).clip(0, 1)
ranking_df["global_hiddenness_component_v1"] = (1.0 - global_commonness).clip(0, 1)

ranking_df["hiddenness_component_v1"] = clip01(
    ranking_df["neighborhood_uniqueness_component_v1"] * 0.60
    + ranking_df["global_hiddenness_component_v1"] * 0.40
)

# Penalizaciones por ruido
ranking_df["negative_penalty_component_v1"] = clip01(
    ranking_df["negative_ratio"].clip(0, 1) * 0.65
    + (ranking_df["price_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1) * 0.10
    + (ranking_df["service_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1) * 0.10
    + ranking_df["fallback_ratio"].clip(0, 1) * 0.15
)

ranking_df["signal_noise_penalty_v1"] = clip01(
    (ranking_df["low_confidence_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1) * 0.40
    + ranking_df["fallback_ratio"].clip(0, 1) * 0.30
    + (ranking_df["service_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1) * 0.15
    + (ranking_df["price_flag_count"] / ranking_df["mention_count"].replace(0, np.nan)).fillna(0).clip(0, 1) * 0.15
)

# Score base ponderado
ranking_df["hidden_gem_score_raw_v1"] = clip01(
    ranking_df["quality_component_v1"] * 0.42
    + ranking_df["evidence_component_v1"] * 0.24
    + ranking_df["confidence_component_v1"] * 0.18
    + ranking_df["hiddenness_component_v1"] * 0.10
    + ranking_df["neighborhood_uniqueness_component_v1"] * 0.06
)

# Score final con penalizaciones suaves
ranking_df["hidden_gem_score_v1"] = (
    ranking_df["hidden_gem_score_raw_v1"]
    * (1.0 - 0.18 * ranking_df["negative_penalty_component_v1"])
    * (1.0 - 0.10 * ranking_df["signal_noise_penalty_v1"])
    * 100.0
).clip(0, 100).round(4)

display(
    ranking_df[
        [
            "place_name", "display_dish_name_es_v1", "neighborhood_name",
            "mention_count", "review_count", "positive_ratio", "negative_ratio",
            "quality_component_v1", "evidence_component_v1", "confidence_component_v1",
            "hiddenness_component_v1", "negative_penalty_component_v1",
            "hidden_gem_score_v1",
        ]
    ]
    .sort_values("hidden_gem_score_v1", ascending=False)
    .head(30)
)


,place_name,display_dish_name_es_v1,neighborhood_name,mention_count,review_count,positive_ratio,negative_ratio,quality_component_v1,evidence_component_v1,confidence_component_v1,hiddenness_component_v1,negative_penalty_component_v1,hidden_gem_score_v1
3,Pizzería San Pablo,pizza,SAN PABLO D Y E,6,5,1.0000,0.0,0.776415,0.881021,0.834067,0.760340,0.058338,80.1471
0,restaurante asiático shan,sushi,BELLAVISTA,3,3,1.0000,0.0,0.751479,0.643142,0.977500,0.836310,0.000000,78.9556
8,Il Ristorantino Dell'Avvocato Calle Cuna,pizza,ALFALFA,5,4,1.0000,0.0,0.689966,0.793963,0.803600,0.760340,0.050000,74.1349
2,Tarannà,atún,SANTA CATALINA,3,3,1.0000,0.0,0.691781,0.635059,0.941140,0.664449,0.000000,73.8812
1,Il Ristorantino Dell´Avvocato Sevilla,pizza,ENCARNACIÓN-REGINA,9,5,0.7778,0.0,0.696438,0.953082,0.601588,0.760340,0.066660,73.2847
16,Flor de la Caña Café y Ron SR.,tarta de queso,LEON XIII-LOS NARANJOS,3,3,1.0000,0.0,0.671214,0.628262,0.947485,0.653110,0.033333,72.0559
6,BAR EL BUCHITO,carrillada,PALMETE,3,2,1.0000,0.0,0.676663,0.584280,0.965485,0.620555,0.000000,72.0268
10,Bodega Mi Tierra,torrija,SANTA AURELIA-CANTABRICO-ATLANTICO-LA ROMERIA,2,2,1.0000,0.0,0.696788,0.483290,0.977500,0.681737,0.000000,71.2764
5,Taberna Los Terceros,solomillo al whisky,SANTA CATALINA,5,3,1.0000,0.0,0.671613,0.731921,0.799100,0.618673,0.030000,70.9464
15,"Venta Campo Rico ""El Arriero""",tarta de queso,VALDEZORRAS,2,2,1.0000,0.0,0.692415,0.481106,0.962740,0.653110,0.000000,70.4884


In [6]:
# ============================================================
# 06. Selección y tiers del ranking piloto
# ============================================================

def assign_hidden_gem_tier(row) -> str:
    score = float(row.get("hidden_gem_score_v1", 0.0))
    evidence = str(row.get("evidence_tier_v1", ""))
    reviews = float(row.get("review_count", 0.0))
    negative_ratio = float(row.get("negative_ratio", 1.0))
    confidence = float(row.get("avg_sentiment_confidence", 0.0))
    ranking_ready = bool(row.get("ranking_ready_v1", False))

    if not ranking_ready or negative_ratio > 0.45 or confidence < 0.45:
        return "not_selected"

    # Muy restrictivo: requiere evidencia sólida y score alto.
    if (
        score >= 78
        and evidence in {"solid", "strong"}
        and reviews >= 3
        and negative_ratio <= 0.20
    ):
        return "top_hidden_gem"

    # Fuerte: evita llamar strong a señales demasiado pequeñas salvo score extremo.
    if (
        (
            score >= 70
            and evidence in {"solid", "strong"}
            and reviews >= 3
        )
        or (
            score >= 74
            and evidence == "emerging"
            and reviews >= 2
        )
    ) and negative_ratio <= 0.25:
        return "strong_hidden_gem"

    if score >= 62 and reviews >= 2 and negative_ratio <= 0.35:
        return "promising_hidden_gem"

    if score >= 55 and reviews >= 2 and negative_ratio <= 0.40:
        return "exploratory_hidden_gem"

    return "not_selected"


ranking_df["hidden_gem_tier_v1"] = ranking_df.apply(assign_hidden_gem_tier, axis=1)
ranking_df["selected_hidden_gem_v1"] = ranking_df["hidden_gem_tier_v1"] != "not_selected"
ranking_df["ranking_scope_v1"] = RANKING_SCOPE
ranking_df["ranking_version_v1"] = VERSION
ranking_df["is_production_ready_v1"] = False

tier_order = {
    "top_hidden_gem": 1,
    "strong_hidden_gem": 2,
    "promising_hidden_gem": 3,
    "exploratory_hidden_gem": 4,
    "not_selected": 9,
}

ranking_df["hidden_gem_tier_order_v1"] = ranking_df["hidden_gem_tier_v1"].map(tier_order).fillna(9).astype(int)

ranking_df = ranking_df.sort_values(
    [
        "selected_hidden_gem_v1",
        "hidden_gem_tier_order_v1",
        "hidden_gem_score_v1",
        "review_count",
        "mention_count",
    ],
    ascending=[False, True, False, False, False],
).reset_index(drop=True)

selected_df = ranking_df[ranking_df["selected_hidden_gem_v1"]].copy().reset_index(drop=True)
selected_df["hidden_gem_global_rank_v1"] = np.arange(1, len(selected_df) + 1)

# Rankings locales
selected_df = selected_df.sort_values(["district_id", "hidden_gem_score_v1"], ascending=[True, False])
selected_df["rank_in_district_v1"] = selected_df.groupby("district_id", dropna=False).cumcount() + 1

selected_df = selected_df.sort_values(["neighborhood_id", "hidden_gem_score_v1"], ascending=[True, False])
selected_df["rank_in_neighborhood_v1"] = selected_df.groupby("neighborhood_id", dropna=False).cumcount() + 1

selected_df = selected_df.sort_values(["dish_id_v1", "hidden_gem_score_v1"], ascending=[True, False])
selected_df["rank_for_dish_v1"] = selected_df.groupby("dish_id_v1", dropna=False).cumcount() + 1

# Volvemos a orden global final
selected_df = selected_df.sort_values("hidden_gem_global_rank_v1").reset_index(drop=True)

rank_cols = [
    "candidate_pair_id_v1",
    "hidden_gem_global_rank_v1",
    "rank_in_district_v1",
    "rank_in_neighborhood_v1",
    "rank_for_dish_v1",
]

ranking_df = ranking_df.merge(
    selected_df[rank_cols],
    on="candidate_pair_id_v1",
    how="left",
)

ranking_df["hidden_gem_global_rank_v1"] = ranking_df["hidden_gem_global_rank_v1"].fillna(0).astype(int)
ranking_df["rank_in_district_v1"] = ranking_df["rank_in_district_v1"].fillna(0).astype(int)
ranking_df["rank_in_neighborhood_v1"] = ranking_df["rank_in_neighborhood_v1"].fillna(0).astype(int)
ranking_df["rank_for_dish_v1"] = ranking_df["rank_for_dish_v1"].fillna(0).astype(int)

selected_df = ranking_df[ranking_df["selected_hidden_gem_v1"]].sort_values("hidden_gem_global_rank_v1").copy()

print("Distribución de tiers:")
display(ranking_df["hidden_gem_tier_v1"].value_counts().rename_axis("tier").reset_index(name="count"))

print("Top ranking seleccionado:")
display(
    selected_df[
        [
            "hidden_gem_global_rank_v1", "hidden_gem_tier_v1",
            "place_name", "district_name", "neighborhood_name",
            "display_dish_name_es_v1", "mention_count", "review_count",
            "positive_ratio", "negative_ratio", "avg_sentiment_confidence",
            "evidence_tier_v1", "signal_quality_tier_v1", "hidden_gem_score_v1",
        ]
    ].head(40)
)


Distribución de tiers:


,tier,count
0,not_selected,106
1,promising_hidden_gem,72
2,exploratory_hidden_gem,69
3,strong_hidden_gem,7
4,top_hidden_gem,2


Top ranking seleccionado:


,hidden_gem_global_rank_v1,hidden_gem_tier_v1,place_name,district_name,neighborhood_name,display_dish_name_es_v1,mention_count,review_count,positive_ratio,negative_ratio,avg_sentiment_confidence,evidence_tier_v1,signal_quality_tier_v1,hidden_gem_score_v1
0,1,top_hidden_gem,Pizzería San Pablo,San Pablo - Santa Justa,SAN PABLO D Y E,pizza,6,5,1.0000,0.0,0.8350,strong,high_quality_signal,80.1471
1,2,top_hidden_gem,restaurante asiático shan,Bellavista - La Palmera,BELLAVISTA,sushi,3,3,1.0000,0.0,0.9500,solid,usable_signal,78.9556
2,3,strong_hidden_gem,Il Ristorantino Dell'Avvocato Calle Cuna,Casco Antiguo,ALFALFA,pizza,5,4,1.0000,0.0,0.8080,strong,high_quality_signal,74.1349
3,4,strong_hidden_gem,Tarannà,Casco Antiguo,SANTA CATALINA,atún,3,3,1.0000,0.0,0.8692,solid,usable_signal,73.8812
4,5,strong_hidden_gem,Il Ristorantino Dell´Avvocato Sevilla,Casco Antiguo,ENCARNACIÓN-REGINA,pizza,9,5,0.7778,0.0,0.6578,strong,high_quality_signal,73.2847
5,6,strong_hidden_gem,Flor de la Caña Café y Ron SR.,Macarena,LEON XIII-LOS NARANJOS,tarta de queso,3,3,1.0000,0.0,0.8833,solid,usable_signal,72.0559
6,7,strong_hidden_gem,Taberna Los Terceros,Casco Antiguo,SANTA CATALINA,solomillo al whisky,5,3,1.0000,0.0,0.7980,solid,usable_signal,70.9464
7,8,strong_hidden_gem,"Cafeteria mi abuela, churreria",Sur,FELIPE II-LOS DIEZ MANDAMIENTOS,churros,6,5,1.0000,0.0,0.6928,strong,high_quality_signal,70.4390
8,9,strong_hidden_gem,I Love Churros - Cafetería churreria en Ronda de Capuchinos,Macarena,CRUZ ROJA-CAPUCHINOS,churros,8,5,0.8750,0.0,0.6325,strong,high_quality_signal,70.3222
9,10,promising_hidden_gem,BAR EL BUCHITO,Cerro - Amate,PALMETE,carrillada,3,2,1.0000,0.0,0.9233,solid,usable_signal,72.0268


In [7]:
# ============================================================
# 07. Explicaciones legibles del ranking
# ============================================================

def build_explanation(row) -> str:
    dish = row.get("display_dish_name_es_v1", "plato")
    place = row.get("place_name", "local")
    score = float(row.get("hidden_gem_score_v1", 0.0))
    mentions = int(row.get("mention_count", 0))
    reviews = int(row.get("review_count", 0))
    pos = float(row.get("positive_ratio", 0.0)) * 100
    neg = float(row.get("negative_ratio", 0.0)) * 100
    evidence = row.get("evidence_tier_v1", "")
    neighborhood = row.get("neighborhood_name", "")
    district = row.get("district_name", "")
    tier = row.get("hidden_gem_tier_v1", "not_selected")

    parts = [
        f"{dish} en {place} obtiene {score:.1f}/100",
        f"con {mentions} menciones en {reviews} reviews",
        f"{pos:.1f}% positivas y {neg:.1f}% negativas",
        f"evidencia {evidence}",
    ]

    if neighborhood:
        parts.append(f"barrio {neighborhood}")
    if district:
        parts.append(f"distrito {district}")

    if row.get("fallback_ratio", 0) > 0.35:
        parts.append("incluye una parte de señal inferida desde rating de review")
    if row.get("local_context_ratio", 0) >= 0.70:
        parts.append("la mayoría de señales vienen del contexto local de la mención")
    if row.get("global_hiddenness_component_v1", 0) >= 0.60:
        parts.append("plato relativamente menos extendido en el corpus global")
    if tier in {"top_hidden_gem", "strong_hidden_gem"}:
        parts.append("candidato destacado del piloto")

    return "; ".join(parts) + "."

ranking_df["ranking_explanation_v1"] = ranking_df.apply(build_explanation, axis=1)

display(
    ranking_df[ranking_df["selected_hidden_gem_v1"]][
        [
            "hidden_gem_global_rank_v1",
            "hidden_gem_tier_v1",
            "place_name",
            "display_dish_name_es_v1",
            "hidden_gem_score_v1",
            "ranking_explanation_v1",
        ]
    ].head(20)
)


,hidden_gem_global_rank_v1,hidden_gem_tier_v1,place_name,display_dish_name_es_v1,hidden_gem_score_v1,ranking_explanation_v1
0,1,top_hidden_gem,Pizzería San Pablo,pizza,80.1471,pizza en Pizzería San Pablo obtiene 80.1/100; con 6 menciones en 5 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio SAN PABLO D Y E; distrito San Pablo - Sa...
1,2,top_hidden_gem,restaurante asiático shan,sushi,78.9556,sushi en restaurante asiático shan obtiene 79.0/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio BELLAVISTA; distrito Bellavista - ...
2,3,strong_hidden_gem,Il Ristorantino Dell'Avvocato Calle Cuna,pizza,74.1349,pizza en Il Ristorantino Dell'Avvocato Calle Cuna obtiene 74.1/100; con 5 menciones en 4 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio ALFALFA; distrito ...
3,4,strong_hidden_gem,Tarannà,atún,73.8812,atún en Tarannà obtiene 73.9/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio SANTA CATALINA; distrito Casco Antiguo; la mayoría de...
4,5,strong_hidden_gem,Il Ristorantino Dell´Avvocato Sevilla,pizza,73.2847,pizza en Il Ristorantino Dell´Avvocato Sevilla obtiene 73.3/100; con 9 menciones en 5 reviews; 77.8% positivas y 0.0% negativas; evidencia strong; barrio ENCARNACIÓN-REGINA; di...
5,6,strong_hidden_gem,Flor de la Caña Café y Ron SR.,tarta de queso,72.0559,tarta de queso en Flor de la Caña Café y Ron SR. obtiene 72.1/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio LEON XIII-LOS NARANJ...
6,7,strong_hidden_gem,Taberna Los Terceros,solomillo al whisky,70.9464,solomillo al whisky en Taberna Los Terceros obtiene 70.9/100; con 5 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio SANTA CATALINA; distrito ...
7,8,strong_hidden_gem,"Cafeteria mi abuela, churreria",churros,70.4390,"churros en Cafeteria mi abuela, churreria obtiene 70.4/100; con 6 menciones en 5 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio FELIPE II-LOS DIEZ MANDAMI..."
8,9,strong_hidden_gem,I Love Churros - Cafetería churreria en Ronda de Capuchinos,churros,70.3222,churros en I Love Churros - Cafetería churreria en Ronda de Capuchinos obtiene 70.3/100; con 8 menciones en 5 reviews; 87.5% positivas y 0.0% negativas; evidencia strong; barri...
9,10,promising_hidden_gem,BAR EL BUCHITO,carrillada,72.0268,carrillada en BAR EL BUCHITO obtiene 72.0/100; con 3 menciones en 2 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio PALMETE; distrito Cerro - Amate; la mayo...


In [8]:
# ============================================================
# 08. Vistas top por barrio, distrito y plato
# ============================================================

top_by_neighborhood_df = (
    selected_df[selected_df["rank_in_neighborhood_v1"] <= 5]
    .sort_values(["district_name", "neighborhood_name", "rank_in_neighborhood_v1"])
    .copy()
)

top_by_district_df = (
    selected_df[selected_df["rank_in_district_v1"] <= 10]
    .sort_values(["district_name", "rank_in_district_v1"])
    .copy()
)

top_by_dish_df = (
    selected_df[selected_df["rank_for_dish_v1"] <= 10]
    .sort_values(["display_dish_name_es_v1", "rank_for_dish_v1"])
    .copy()
)

print("Top por barrio:", top_by_neighborhood_df.shape)
display(
    top_by_neighborhood_df[
        [
            "district_name", "neighborhood_name", "rank_in_neighborhood_v1",
            "place_name", "display_dish_name_es_v1",
            "hidden_gem_tier_v1", "hidden_gem_score_v1",
            "mention_count", "review_count", "positive_ratio",
        ]
    ].head(50)
)

print("Top por distrito:", top_by_district_df.shape)
display(
    top_by_district_df[
        [
            "district_name", "rank_in_district_v1",
            "place_name", "neighborhood_name", "display_dish_name_es_v1",
            "hidden_gem_tier_v1", "hidden_gem_score_v1",
            "mention_count", "review_count", "positive_ratio",
        ]
    ].head(50)
)


Top por barrio: (141, 87)


,district_name,neighborhood_name,rank_in_neighborhood_v1,place_name,display_dish_name_es_v1,hidden_gem_tier_v1,hidden_gem_score_v1,mention_count,review_count,positive_ratio
1,Bellavista - La Palmera,BELLAVISTA,1,restaurante asiático shan,sushi,top_hidden_gem,78.9556,3,3,1.0000
137,Bellavista - La Palmera,BELLAVISTA,2,Restaurante Reverde Bar & Coffee,arepa,exploratory_hidden_gem,55.6307,2,2,1.0000
63,Bellavista - La Palmera,ELCANO-BERMEJALES,1,Bar - Terraza La Cucamona,tarta de queso,promising_hidden_gem,64.8357,2,2,1.0000
72,Bellavista - La Palmera,ELCANO-BERMEJALES,2,Infanta Sevilla,tarta de queso,promising_hidden_gem,63.0548,2,2,1.0000
55,Bellavista - La Palmera,SECTOR SUR-LA PALMERA-REINA MERCEDES,1,Bodega Asador La Pitarra Heliopolis,solomillo al whisky,promising_hidden_gem,65.5482,2,2,1.0000
64,Bellavista - La Palmera,SECTOR SUR-LA PALMERA-REINA MERCEDES,2,"Duas Terras, El português",bacalao,promising_hidden_gem,64.2872,2,2,1.0000
144,Bellavista - La Palmera,SECTOR SUR-LA PALMERA-REINA MERCEDES,3,Bar Metropolis 1985,croqueta de jamón,exploratory_hidden_gem,55.2565,2,2,1.0000
2,Casco Antiguo,ALFALFA,1,Il Ristorantino Dell'Avvocato Calle Cuna,pizza,strong_hidden_gem,74.1349,5,4,1.0000
49,Casco Antiguo,ALFALFA,2,El Sembrao,torrija,promising_hidden_gem,66.2721,3,3,1.0000
71,Casco Antiguo,ALFALFA,3,El Sembrao,atún,promising_hidden_gem,63.1094,4,4,1.0000


Top por distrito: (98, 87)


,district_name,rank_in_district_v1,place_name,neighborhood_name,display_dish_name_es_v1,hidden_gem_tier_v1,hidden_gem_score_v1,mention_count,review_count,positive_ratio
1,Bellavista - La Palmera,1,restaurante asiático shan,BELLAVISTA,sushi,top_hidden_gem,78.9556,3,3,1.0000
55,Bellavista - La Palmera,2,Bodega Asador La Pitarra Heliopolis,SECTOR SUR-LA PALMERA-REINA MERCEDES,solomillo al whisky,promising_hidden_gem,65.5482,2,2,1.0000
63,Bellavista - La Palmera,3,Bar - Terraza La Cucamona,ELCANO-BERMEJALES,tarta de queso,promising_hidden_gem,64.8357,2,2,1.0000
64,Bellavista - La Palmera,4,"Duas Terras, El português",SECTOR SUR-LA PALMERA-REINA MERCEDES,bacalao,promising_hidden_gem,64.2872,2,2,1.0000
72,Bellavista - La Palmera,5,Infanta Sevilla,ELCANO-BERMEJALES,tarta de queso,promising_hidden_gem,63.0548,2,2,1.0000
137,Bellavista - La Palmera,6,Restaurante Reverde Bar & Coffee,BELLAVISTA,arepa,exploratory_hidden_gem,55.6307,2,2,1.0000
144,Bellavista - La Palmera,7,Bar Metropolis 1985,SECTOR SUR-LA PALMERA-REINA MERCEDES,croqueta de jamón,exploratory_hidden_gem,55.2565,2,2,1.0000
2,Casco Antiguo,1,Il Ristorantino Dell'Avvocato Calle Cuna,ALFALFA,pizza,strong_hidden_gem,74.1349,5,4,1.0000
3,Casco Antiguo,2,Tarannà,SANTA CATALINA,atún,strong_hidden_gem,73.8812,3,3,1.0000
4,Casco Antiguo,3,Il Ristorantino Dell´Avvocato Sevilla,ENCARNACIÓN-REGINA,pizza,strong_hidden_gem,73.2847,9,5,0.7778


In [9]:
# ============================================================
# 09. Resúmenes agregados del ranking
# ============================================================

tier_summary_df = (
    ranking_df
    .groupby("hidden_gem_tier_v1", dropna=False)
    .agg(
        pair_count=("candidate_pair_id_v1", "count"),
        selected_count=("selected_hidden_gem_v1", "sum"),
        avg_score=("hidden_gem_score_v1", "mean"),
        min_score=("hidden_gem_score_v1", "min"),
        max_score=("hidden_gem_score_v1", "max"),
        avg_mentions=("mention_count", "mean"),
        avg_reviews=("review_count", "mean"),
    )
    .reset_index()
    .sort_values("hidden_gem_tier_v1")
)

district_summary_df = (
    selected_df
    .groupby(["district_id", "district_name"], dropna=False)
    .agg(
        selected_hidden_gem_count=("candidate_pair_id_v1", "count"),
        unique_places=("place_id", "nunique"),
        unique_dishes=("dish_id_v1", "nunique"),
        avg_score=("hidden_gem_score_v1", "mean"),
        max_score=("hidden_gem_score_v1", "max"),
        avg_mentions=("mention_count", "mean"),
        avg_reviews=("review_count", "mean"),
    )
    .reset_index()
    .sort_values(["selected_hidden_gem_count", "max_score"], ascending=[False, False])
)

neighborhood_summary_df = (
    selected_df
    .groupby(["district_id", "district_name", "neighborhood_id", "neighborhood_name"], dropna=False)
    .agg(
        selected_hidden_gem_count=("candidate_pair_id_v1", "count"),
        unique_places=("place_id", "nunique"),
        unique_dishes=("dish_id_v1", "nunique"),
        avg_score=("hidden_gem_score_v1", "mean"),
        max_score=("hidden_gem_score_v1", "max"),
        avg_mentions=("mention_count", "mean"),
        avg_reviews=("review_count", "mean"),
    )
    .reset_index()
    .sort_values(["selected_hidden_gem_count", "max_score"], ascending=[False, False])
)

dish_summary_df = (
    selected_df
    .groupby(["dish_id_v1", "display_dish_name_es_v1", "normalized_dish_name_es_v1", "dish_family_es_v1"], dropna=False)
    .agg(
        selected_hidden_gem_count=("candidate_pair_id_v1", "count"),
        unique_places=("place_id", "nunique"),
        unique_neighborhoods=("neighborhood_id", "nunique"),
        avg_score=("hidden_gem_score_v1", "mean"),
        max_score=("hidden_gem_score_v1", "max"),
        avg_mentions=("mention_count", "mean"),
        avg_reviews=("review_count", "mean"),
    )
    .reset_index()
    .sort_values(["selected_hidden_gem_count", "max_score"], ascending=[False, False])
)

display(tier_summary_df)
display(district_summary_df.head(20))
display(neighborhood_summary_df.head(20))
display(dish_summary_df.head(30))


,hidden_gem_tier_v1,pair_count,selected_count,avg_score,min_score,max_score,avg_mentions,avg_reviews
0,exploratory_hidden_gem,69,69,58.403862,55.0423,61.7183,2.855072,2.463768
1,not_selected,106,0,50.744708,38.4656,54.9792,2.415094,2.122642
2,promising_hidden_gem,72,72,66.639331,62.1400,72.0268,2.513889,2.347222
3,strong_hidden_gem,7,7,72.152043,70.3222,74.1349,5.571429,4.000000
4,top_hidden_gem,2,2,79.551350,78.9556,80.1471,4.500000,4.000000


,district_id,district_name,selected_hidden_gem_count,unique_places,unique_dishes,avg_score,max_score,avg_mentions,avg_reviews
3,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,44,36,22,64.508841,74.1349,2.795455,2.477273
0,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,15,11,9,64.262633,72.0268,2.933333,2.733333
2,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,14,10,11,63.190300,70.4390,3.000000,2.642857
5,7f42c0b3-14d1-400c-bcde-c128faafc4c5,Este - Alcosa - Torreblanca,14,10,11,60.835164,68.1937,3.071429,2.500000
4,6af93f2e-b5da-4456-9cb1-e68958a97dbf,Nervión,13,10,10,60.801623,70.1081,2.692308,2.307692
6,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,12,11,11,64.263167,80.1471,2.583333,2.333333
1,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,10,10,8,63.499520,72.0559,3.900000,3.300000
9,f9420edb-f021-42df-a2cc-e4bb21d0eceb,Triana,10,8,9,61.983450,69.4382,2.600000,2.100000
7,af2fbf25-5491-4c27-811f-28b7a400ea84,Norte,9,7,9,62.703422,70.4884,2.666667,2.444444
8,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,7,7,6,63.938386,78.9556,2.142857,2.142857


,district_id,district_name,neighborhood_id,neighborhood_name,selected_hidden_gem_count,unique_places,unique_dishes,avg_score,max_score,avg_mentions,avg_reviews
28,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,dd0845a6-120f-4945-9a3d-ec166c36337c,FERIA,8,5,7,66.544162,69.6970,2.750000,2.375000
24,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,8dea766c-0cd6-499c-b34c-2fed439f58df,SANTA CRUZ,7,7,7,63.503957,69.3353,2.428571,2.285714
3,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,7feeccb6-1f5d-47c2-a067-02dc6466c762,EL CERRO,6,3,6,63.389917,69.7490,2.833333,2.666667
14,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,4cc93289-8e5d-4379-8ca1-74cacb5f263a,EL PORVENIR,6,3,4,62.214800,69.5217,2.833333,2.500000
29,6af93f2e-b5da-4456-9cb1-e68958a97dbf,Nervión,00162efb-57bb-451d-a462-2bad40efc8b0,LA BUHAIRA,6,4,6,60.231000,68.5351,2.333333,2.333333
34,7f42c0b3-14d1-400c-bcde-c128faafc4c5,Este - Alcosa - Torreblanca,4741768d-f63a-4d4a-8344-d7aebb0e3303,PARQUE ALCOSA-JARDINES DEL EDÉN,6,3,6,63.204967,68.1937,2.666667,2.000000
20,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,282de85f-269a-4337-a3b8-75d8ac5fa1ea,ALFALFA,5,3,5,64.695100,74.1349,3.400000,3.200000
53,f9420edb-f021-42df-a2cc-e4bb21d0eceb,Triana,b60943da-ab5b-480a-9613-54f47f9dda3b,TRIANA CASCO ANTIGUO,5,5,5,59.400980,68.6300,2.800000,2.200000
19,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,0c02556b-4c80-4a87-876b-bcf430924273,MUSEO,5,4,5,62.091120,63.7085,2.600000,2.400000
25,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,903feb9b-aed1-4717-abbf-e693ac5585a0,SANTA CATALINA,4,4,4,67.978375,73.8812,3.750000,3.250000


,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,selected_hidden_gem_count,unique_places,unique_neighborhoods,avg_score,max_score,avg_mentions,avg_reviews
2,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,16,16,16,61.874731,69.5217,2.500000,2.187500
4,dish_es_v1_18ef91885a7312e4,tarta de queso,tarta de queso,postres_y_desayunos,12,12,10,65.399583,72.0559,2.500000,2.250000
6,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,10,10,10,66.910430,80.1471,4.700000,3.500000
8,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,10,10,10,63.379280,72.0268,2.800000,2.400000
24,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,9,9,8,65.507589,73.8812,3.333333,2.888889
1,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,tapas_clasicas,9,9,9,63.856756,70.9464,2.444444,2.222222
0,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,mar_y_pescado,9,9,9,60.590511,67.4108,2.111111,2.000000
32,dish_es_v1_da687fa2af0b4205,churros,churros,postres_y_desayunos,7,7,7,64.350714,70.4390,5.000000,4.428571
30,dish_es_v1_d130c191c0bc5a8e,torrija,torrija,postres_y_desayunos,6,6,6,64.054067,71.2764,2.166667,2.166667
19,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,5,5,4,61.189360,68.1937,3.000000,2.800000


In [10]:
# ============================================================
# 10. Muestra para revisión manual
# ============================================================

manual_review_df = (
    ranking_df[
        (ranking_df["selected_hidden_gem_v1"])
        | (ranking_df["hidden_gem_score_v1"] >= ranking_df["hidden_gem_score_v1"].quantile(0.80))
        | (ranking_df["negative_ratio"] >= 0.30)
        | (ranking_df["fallback_ratio"] >= 0.50)
    ]
    .sort_values(
        ["selected_hidden_gem_v1", "hidden_gem_score_v1", "negative_ratio"],
        ascending=[False, False, False]
    )
    .head(250)
    .copy()
)

manual_columns = [
    "hidden_gem_global_rank_v1",
    "hidden_gem_tier_v1",
    "selected_hidden_gem_v1",
    "place_name",
    "district_name",
    "neighborhood_name",
    "display_dish_name_es_v1",
    "mention_count",
    "review_count",
    "positive_ratio",
    "negative_ratio",
    "smoothed_positive_ratio",
    "smoothed_negative_ratio",
    "weighted_sentiment_score",
    "avg_sentiment_confidence",
    "local_context_ratio",
    "fallback_ratio",
    "evidence_tier_v1",
    "signal_quality_tier_v1",
    "quality_component_v1",
    "evidence_component_v1",
    "confidence_component_v1",
    "hiddenness_component_v1",
    "negative_penalty_component_v1",
    "signal_noise_penalty_v1",
    "hidden_gem_score_v1",
    "example_positive_context_v1",
    "example_negative_context_v1",
    "example_neutral_context_v1",
    "ranking_explanation_v1",
]

manual_columns = [c for c in manual_columns if c in manual_review_df.columns]
display(manual_review_df[manual_columns].head(80))


,hidden_gem_global_rank_v1,hidden_gem_tier_v1,selected_hidden_gem_v1,place_name,district_name,neighborhood_name,display_dish_name_es_v1,mention_count,review_count,positive_ratio,negative_ratio,smoothed_positive_ratio,smoothed_negative_ratio,weighted_sentiment_score,avg_sentiment_confidence,local_context_ratio,fallback_ratio,evidence_tier_v1,signal_quality_tier_v1,quality_component_v1,evidence_component_v1,confidence_component_v1,hiddenness_component_v1,negative_penalty_component_v1,signal_noise_penalty_v1,hidden_gem_score_v1,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,ranking_explanation_v1
0,1,top_hidden_gem,True,Pizzería San Pablo,San Pablo - Santa Justa,SAN PABLO D Y E,pizza,6,5,1.0000,0.0,0.7301,0.1350,5.3569,0.8350,0.8333,0.1667,strong,high_quality_signal,0.776415,0.881021,0.834067,0.760340,0.058338,0.166677,80.1471,"Unas experiencia muy buena la pizza 10 de 10 me encanto, esta vez fue la carnivora y seguiré probando las demás pizza q tienen, 100 % recomendados",,,pizza en Pizzería San Pablo obtiene 80.1/100; con 6 menciones en 5 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio SAN PABLO D Y E; distrito San Pablo - Sa...
1,2,top_hidden_gem,True,restaurante asiático shan,Bellavista - La Palmera,BELLAVISTA,sushi,3,3,1.0000,0.0,0.6581,0.1709,4.9497,0.9500,1.0000,0.0000,solid,usable_signal,0.751479,0.643142,0.977500,0.836310,0.000000,0.000000,78.9556,"Comida riquísima, el sushi flambeado y el yakisoba muy bueno.",,,sushi en restaurante asiático shan obtiene 79.0/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio BELLAVISTA; distrito Bellavista - ...
2,3,strong_hidden_gem,True,Il Ristorantino Dell'Avvocato Calle Cuna,Casco Antiguo,ALFALFA,pizza,5,4,1.0000,0.0,0.6943,0.1529,2.7308,0.8080,0.8000,0.2000,strong,high_quality_signal,0.689966,0.793963,0.803600,0.760340,0.050000,0.170000,74.1349,"Muy rico todo, probamos las pizzas que tienen en carta y fuera de carta, buenísima calidad y sabor.",,,pizza en Il Ristorantino Dell'Avvocato Calle Cuna obtiene 74.1/100; con 5 menciones en 4 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio ALFALFA; distrito ...
3,4,strong_hidden_gem,True,Tarannà,Casco Antiguo,SANTA CATALINA,atún,3,3,1.0000,0.0,0.6433,0.1783,3.3604,0.8692,1.0000,0.0000,solid,usable_signal,0.691781,0.635059,0.941140,0.664449,0.000000,0.000000,73.8812,Tomamos un tartar de atún de almadraba súper recomendable y un arroz súper jugoso.,,,atún en Tarannà obtiene 73.9/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio SANTA CATALINA; distrito Casco Antiguo; la mayoría de...
4,5,strong_hidden_gem,True,Il Ristorantino Dell´Avvocato Sevilla,Casco Antiguo,ENCARNACIÓN-REGINA,pizza,9,5,0.7778,0.0,0.7293,0.1248,2.4826,0.6578,0.5556,0.4444,strong,high_quality_signal,0.696438,0.953082,0.601588,0.760340,0.066660,0.311098,73.2847,"Como platos principales pedimos pizzas, y fue todo un acierto: una fuera de carta con guiso de carne, increíblemente sabrosa y diferente, y otra de carta, ahumada, con un sabor...",,"Llegamos con ganas de comer una pizza y una bola de arroz rellena de carne y salsa, para nuestra sorpresa nos sentamos en mesa y nos dicen que solo hacen pizza, igual habrá que...",pizza en Il Ristorantino Dell´Avvocato Sevilla obtiene 73.3/100; con 9 menciones en 5 reviews; 77.8% positivas y 0.0% negativas; evidencia strong; barrio ENCARNACIÓN-REGINA; di...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,76,promising_hidden_gem,True,BOCALLENA,Sur,HUERTA DE LA SALUD,ensaladilla,2,2,0.5000,0.0,0.4362,0.2237,2.5773,0.7350,1.0000,0.0000,emerging,emerging_signal,0.560894,0.468956,0.880750,0.603499,0.000000,0.000000,62.7010,"He ido hoy por primera vez a este sitio nuevo en el barrio Cerveza de 10, de las mejores que he tomado últimamente Ensaladilla atún top también Albóndigas de choco muy ricas, q...",

In [11]:
# ============================================================
# 11. Guardado de artefactos
# ============================================================

all_candidates_csv = OUTPUT_DIR / "sevilla_hidden_gems_candidates_all_v1.csv"
all_candidates_jsonl = OUTPUT_DIR / "sevilla_hidden_gems_candidates_all_v1.jsonl"

selected_csv = OUTPUT_DIR / "sevilla_hidden_gems_selected_candidates_v1.csv"
selected_jsonl = OUTPUT_DIR / "sevilla_hidden_gems_selected_candidates_v1.jsonl"

top_by_neighborhood_csv = OUTPUT_DIR / "sevilla_hidden_gems_top_by_neighborhood_v1.csv"
top_by_district_csv = OUTPUT_DIR / "sevilla_hidden_gems_top_by_district_v1.csv"
top_by_dish_csv = OUTPUT_DIR / "sevilla_hidden_gems_top_by_dish_v1.csv"

tier_summary_csv = OUTPUT_DIR / "sevilla_hidden_gems_tier_summary_v1.csv"
district_summary_csv = OUTPUT_DIR / "sevilla_hidden_gems_district_summary_v1.csv"
neighborhood_summary_csv = OUTPUT_DIR / "sevilla_hidden_gems_neighborhood_summary_v1.csv"
dish_summary_csv = OUTPUT_DIR / "sevilla_hidden_gems_dish_summary_v1.csv"

manual_review_csv = OUTPUT_DIR / "sevilla_hidden_gems_manual_review_v1.csv"
summary_json = OUTPUT_DIR / "sevilla_hidden_gems_ranking_summary_v1.json"

ranking_df.to_csv(all_candidates_csv, index=False, encoding="utf-8")
save_jsonl(ranking_df, all_candidates_jsonl)

selected_df.to_csv(selected_csv, index=False, encoding="utf-8")
save_jsonl(selected_df, selected_jsonl)

top_by_neighborhood_df.to_csv(top_by_neighborhood_csv, index=False, encoding="utf-8")
top_by_district_df.to_csv(top_by_district_csv, index=False, encoding="utf-8")
top_by_dish_df.to_csv(top_by_dish_csv, index=False, encoding="utf-8")

tier_summary_df.to_csv(tier_summary_csv, index=False, encoding="utf-8")
district_summary_df.to_csv(district_summary_csv, index=False, encoding="utf-8")
neighborhood_summary_df.to_csv(neighborhood_summary_csv, index=False, encoding="utf-8")
dish_summary_df.to_csv(dish_summary_csv, index=False, encoding="utf-8")

manual_review_df.to_csv(manual_review_csv, index=False, encoding="utf-8")

print("Artefactos guardados en:", OUTPUT_DIR)
for p in [
    all_candidates_csv,
    all_candidates_jsonl,
    selected_csv,
    selected_jsonl,
    top_by_neighborhood_csv,
    top_by_district_csv,
    top_by_dish_csv,
    tier_summary_csv,
    district_summary_csv,
    neighborhood_summary_csv,
    dish_summary_csv,
    manual_review_csv,
]:
    print("-", p.name, "->", p)


Artefactos guardados en: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking
- sevilla_hidden_gems_candidates_all_v1.csv -> c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking\sevilla_hidden_gems_candidates_all_v1.csv
- sevilla_hidden_gems_candidates_all_v1.jsonl -> c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking\sevilla_hidden_gems_candidates_all_v1.jsonl
- sevilla_hidden_gems_selected_candidates_v1.csv -> c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking\sevilla_hidden_gems_selected_candidates_v1.csv
- sevilla_hidden_gems_selected_candidates_v1.jsonl -> c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking\sevilla_hidden_gems_selected_candidates_v1.jsonl
- sevilla_hidden_gems_top_by_neighbo

In [12]:
# ============================================================
# 12. Resumen JSON y checks finales
# ============================================================

# Aseguramos columnas finales mínimas por si alguna celda previa no se ejecutó o
# si se ejecutó una versión anterior del notebook.
if len(selected_df):
    selected_df = selected_df.copy()

    if "hidden_gem_global_rank_v1" not in selected_df.columns:
        selected_df = selected_df.sort_values(
            ["hidden_gem_score_v1", "review_count", "mention_count"],
            ascending=[False, False, False],
        ).reset_index(drop=True)
        selected_df["hidden_gem_global_rank_v1"] = np.arange(1, len(selected_df) + 1)

    if "is_production_ready_v1" not in selected_df.columns:
        selected_df["is_production_ready_v1"] = False

    if "ranking_scope_v1" not in selected_df.columns:
        selected_df["ranking_scope_v1"] = RANKING_SCOPE

    if "ranking_explanation_v1" not in selected_df.columns:
        selected_df["ranking_explanation_v1"] = selected_df.apply(build_explanation, axis=1)

# Reinyectamos columnas corregidas en ranking_df si existen.
if len(selected_df) and "candidate_pair_id_v1" in selected_df.columns and "candidate_pair_id_v1" in ranking_df.columns:
    refresh_cols = [
        "candidate_pair_id_v1",
        "hidden_gem_global_rank_v1",
        "ranking_explanation_v1",
        "is_production_ready_v1",
        "ranking_scope_v1",
    ]
    refresh_cols = [c for c in refresh_cols if c in selected_df.columns]
    ranking_df = ranking_df.drop(
        columns=[c for c in refresh_cols if c != "candidate_pair_id_v1" and c in ranking_df.columns],
        errors="ignore",
    ).merge(
        selected_df[refresh_cols],
        on="candidate_pair_id_v1",
        how="left",
    )
    if "is_production_ready_v1" in ranking_df.columns:
        ranking_df["is_production_ready_v1"] = ranking_df["is_production_ready_v1"].fillna(False).astype(bool)
    if "ranking_scope_v1" in ranking_df.columns:
        ranking_df["ranking_scope_v1"] = ranking_df["ranking_scope_v1"].fillna(RANKING_SCOPE)

selected_score_summary = (
    selected_df["hidden_gem_score_v1"].describe().round(4).to_dict()
    if len(selected_df) and "hidden_gem_score_v1" in selected_df.columns
    else {}
)

desired_top_30_columns = [
    "hidden_gem_global_rank_v1",
    "hidden_gem_tier_v1",
    "place_id",
    "place_name",
    "district_name",
    "neighborhood_name",
    "dish_id_v1",
    "display_dish_name_es_v1",
    "mention_count",
    "review_count",
    "positive_ratio",
    "negative_ratio",
    "avg_sentiment_confidence",
    "evidence_tier_v1",
    "signal_quality_tier_v1",
    "hidden_gem_score_v1",
    "ranking_explanation_v1",
]

available_top_30_columns = [c for c in desired_top_30_columns if c in selected_df.columns]
missing_top_30_columns = [c for c in desired_top_30_columns if c not in selected_df.columns]

if missing_top_30_columns:
    print("Columnas no disponibles en top_30:", missing_top_30_columns)

top_30_records = (
    selected_df[available_top_30_columns].head(30).to_dict(orient="records")
    if len(selected_df) and available_top_30_columns
    else []
)

summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "ranking_scope": RANKING_SCOPE,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "input": {
        "candidate_pairs": int(len(candidates_df)),
        "global_dishes": int(len(global_dish_df)),
        "ranking_ready_input_pairs": int(coerce_bool_series(candidates_df["ranking_ready_v1"]).sum())
            if "ranking_ready_v1" in candidates_df.columns else 0,
    },
    "ranking": {
        "total_pairs_scored": int(len(ranking_df)),
        "selected_hidden_gem_candidates": int(len(selected_df)),
        "selected_places": int(selected_df["place_id"].nunique()) if len(selected_df) and "place_id" in selected_df.columns else 0,
        "selected_dishes": int(selected_df["dish_id_v1"].nunique()) if len(selected_df) and "dish_id_v1" in selected_df.columns else 0,
        "selected_neighborhoods": int(selected_df["neighborhood_id"].nunique()) if len(selected_df) and "neighborhood_id" in selected_df.columns else 0,
        "selected_districts": int(selected_df["district_id"].nunique()) if len(selected_df) and "district_id" in selected_df.columns else 0,
        "tier_counts": ranking_df["hidden_gem_tier_v1"].value_counts().to_dict()
            if "hidden_gem_tier_v1" in ranking_df.columns else {},
        "score_summary_all": ranking_df["hidden_gem_score_v1"].describe().round(4).to_dict()
            if len(ranking_df) and "hidden_gem_score_v1" in ranking_df.columns else {},
        "score_summary_selected": selected_score_summary,
        "top_30": top_30_records,
    },
    "component_weights": {
        "quality_component": 0.42,
        "evidence_component": 0.24,
        "confidence_component": 0.18,
        "hiddenness_component": 0.10,
        "neighborhood_uniqueness_component": 0.06,
        "negative_penalty_max_weight": 0.18,
        "signal_noise_penalty_max_weight": 0.10,
    },
    "checks": {
        "has_candidates": bool(len(ranking_df) > 0),
        "has_selected_candidates": bool(len(selected_df) > 0),
        "score_in_0_100": bool(ranking_df["hidden_gem_score_v1"].between(0, 100).all())
            if len(ranking_df) and "hidden_gem_score_v1" in ranking_df.columns else False,
        "selected_have_neighborhood": bool(selected_df["neighborhood_id"].notna().all())
            if len(selected_df) and "neighborhood_id" in selected_df.columns else True,
        "selected_have_district": bool(selected_df["district_id"].notna().all())
            if len(selected_df) and "district_id" in selected_df.columns else True,
        "selected_have_place": bool(selected_df["place_id"].notna().all())
            if len(selected_df) and "place_id" in selected_df.columns else True,
        "selected_have_dish": bool(selected_df["dish_id_v1"].notna().all())
            if len(selected_df) and "dish_id_v1" in selected_df.columns else True,
        "global_ranks_are_unique": bool(
            selected_df["hidden_gem_global_rank_v1"].nunique(dropna=True) == len(selected_df)
        ) if len(selected_df) and "hidden_gem_global_rank_v1" in selected_df.columns else True,
        "all_selected_are_not_production_ready": bool((selected_df["is_production_ready_v1"] == False).all())
            if len(selected_df) and "is_production_ready_v1" in selected_df.columns else True,
        "output_all_csv_exists": bool(all_candidates_csv.exists()),
        "output_selected_csv_exists": bool(selected_csv.exists()),
        "output_selected_jsonl_exists": bool(selected_jsonl.exists()),
    },
    "artifacts": {
        "all_candidates_csv": all_candidates_csv.name,
        "all_candidates_jsonl": all_candidates_jsonl.name,
        "selected_candidates_csv": selected_csv.name,
        "selected_candidates_jsonl": selected_jsonl.name,
        "top_by_neighborhood_csv": top_by_neighborhood_csv.name,
        "top_by_district_csv": top_by_district_csv.name,
        "top_by_dish_csv": top_by_dish_csv.name,
        "tier_summary_csv": tier_summary_csv.name,
        "district_summary_csv": district_summary_csv.name,
        "neighborhood_summary_csv": neighborhood_summary_csv.name,
        "dish_summary_csv": dish_summary_csv.name,
        "manual_review_csv": manual_review_csv.name,
    },
    "notes": [
        "Ranking piloto basado en reviews de Google Places y señales híbridas de NLP/reglas.",
        "No se marca como producción: is_production_ready_v1 = false.",
        "La evidencia por local está limitada por el número de reviews disponibles por lugar.",
        "Debe revisarse manualmente una muestra antes de cargarlo como ranking operativo.",
    ],
}

# Guardamos de nuevo los artefactos principales con columnas corregidas, si procede.
ranking_df.to_csv(all_candidates_csv, index=False, encoding="utf-8")
save_jsonl(ranking_df, all_candidates_jsonl)
selected_df.to_csv(selected_csv, index=False, encoding="utf-8")
save_jsonl(selected_df, selected_jsonl)

# JSON estricto: sin NaN, Infinity ni tipos numpy/pandas.
safe_json_dump(summary, summary_json)

print(json.dumps(to_builtin(summary["checks"]), indent=2, ensure_ascii=False, allow_nan=False))
print("Summary guardado y validado en:", summary_json)
print("Top 30 registros en summary:", len(summary["ranking"]["top_30"]))

display(pd.DataFrame(summary["ranking"]["top_30"]))


{
  "has_candidates": true,
  "has_selected_candidates": true,
  "score_in_0_100": true,
  "selected_have_neighborhood": true,
  "selected_have_district": true,
  "selected_have_place": true,
  "selected_have_dish": true,
  "global_ranks_are_unique": true,
  "all_selected_are_not_production_ready": true,
  "output_all_csv_exists": true,
  "output_selected_csv_exists": true,
  "output_selected_jsonl_exists": true
}
Summary guardado y validado en: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\ranking\sevilla_hidden_gems_ranking_summary_v1.json
Top 30 registros en summary: 30


,hidden_gem_global_rank_v1,hidden_gem_tier_v1,place_id,place_name,district_name,neighborhood_name,dish_id_v1,display_dish_name_es_v1,mention_count,review_count,positive_ratio,negative_ratio,avg_sentiment_confidence,evidence_tier_v1,signal_quality_tier_v1,hidden_gem_score_v1,ranking_explanation_v1
0,1,top_hidden_gem,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,San Pablo - Santa Justa,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,6,5,1.0000,0.0,0.8350,strong,high_quality_signal,80.1471,pizza en Pizzería San Pablo obtiene 80.1/100; con 6 menciones en 5 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio SAN PABLO D Y E; distrito San Pablo - Sa...
1,2,top_hidden_gem,52d0c5c7-da1f-4733-a8a1-6d3fcac2d18d,restaurante asiático shan,Bellavista - La Palmera,BELLAVISTA,dish_es_v1_451dcf9913bf3b32,sushi,3,3,1.0000,0.0,0.9500,solid,usable_signal,78.9556,sushi en restaurante asiático shan obtiene 79.0/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio BELLAVISTA; distrito Bellavista - ...
2,3,strong_hidden_gem,56979254-f513-4d56-84c0-bf27dd97806e,Il Ristorantino Dell'Avvocato Calle Cuna,Casco Antiguo,ALFALFA,dish_es_v1_1f6ccd2be75f1cc9,pizza,5,4,1.0000,0.0,0.8080,strong,high_quality_signal,74.1349,pizza en Il Ristorantino Dell'Avvocato Calle Cuna obtiene 74.1/100; con 5 menciones en 4 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio ALFALFA; distrito ...
3,4,strong_hidden_gem,a3db8f79-3216-4c52-8307-7730e9335737,Tarannà,Casco Antiguo,SANTA CATALINA,dish_es_v1_ad3755cf6fd0fb67,atún,3,3,1.0000,0.0,0.8692,solid,usable_signal,73.8812,atún en Tarannà obtiene 73.9/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio SANTA CATALINA; distrito Casco Antiguo; la mayoría de...
4,5,strong_hidden_gem,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,Casco Antiguo,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,9,5,0.7778,0.0,0.6578,strong,high_quality_signal,73.2847,pizza en Il Ristorantino Dell´Avvocato Sevilla obtiene 73.3/100; con 9 menciones en 5 reviews; 77.8% positivas y 0.0% negativas; evidencia strong; barrio ENCARNACIÓN-REGINA; di...
5,6,strong_hidden_gem,c93f6669-5e46-4079-b296-b6bb6410a0e3,Flor de la Caña Café y Ron SR.,Macarena,LEON XIII-LOS NARANJOS,dish_es_v1_18ef91885a7312e4,tarta de queso,3,3,1.0000,0.0,0.8833,solid,usable_signal,72.0559,tarta de queso en Flor de la Caña Café y Ron SR. obtiene 72.1/100; con 3 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio LEON XIII-LOS NARANJ...
6,7,strong_hidden_gem,1aca2d21-7dbe-4269-b360-889b689724e9,Taberna Los Terceros,Casco Antiguo,SANTA CATALINA,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,5,3,1.0000,0.0,0.7980,solid,usable_signal,70.9464,solomillo al whisky en Taberna Los Terceros obtiene 70.9/100; con 5 menciones en 3 reviews; 100.0% positivas y 0.0% negativas; evidencia solid; barrio SANTA CATALINA; distrito ...
7,8,strong_hidden_gem,a268a2e5-a430-4b13-9429-c7a356709b1b,"Cafeteria mi abuela, churreria",Sur,FELIPE II-LOS DIEZ MANDAMIENTOS,dish_es_v1_da687fa2af0b4205,churros,6,5,1.0000,0.0,0.6928,strong,high_quality_signal,70.4390,"churros en Cafeteria mi abuela, churreria obtiene 70.4/100; con 6 menciones en 5 reviews; 100.0% positivas y 0.0% negativas; evidencia strong; barrio FELIPE II-LOS DIEZ MANDAMI..."
8,9,strong_hidden_gem,6ee0277a-c373-4d0d-94bc-f7d90766cdc2,I Love Churros - Cafetería churreria en Ronda de Capuchinos,Macarena,CRUZ ROJA-CAPUCHINOS,dish_es_v1_da687fa2af0b4205,churros,8,5,0.8750,0.0,0.6325,strong,high_quality_signal,70.3222,churros en I Love Churros - Cafetería churreria en Ronda de Capuchinos obtiene 70.3/100; con 8 menciones en 5 reviews; 87.5% positivas y 0.0% negativas; evidencia strong; barri...
9,10,promising_hidden_gem,2cf94026-9350-4b2c-abf1-54a08af310d3,BAR EL BUCHITO,Cerro - Amate,PALMETE,dish_es_v1_2c67f67ce386e9d6,carrillada,3,2,1.0000,0.0,0.9233,solid,usable_s

## 13. Próximo paso

Si el resumen y las muestras son coherentes, el siguiente paso será preparar la carga en PostgreSQL como:

- `ranking_scope = "sevilla_pilot"`
- `is_production_ready = false`

Antes de cargar, conviene revisar manualmente:

- `sevilla_hidden_gems_manual_review_v1.csv`
- `sevilla_hidden_gems_selected_candidates_v1.csv`
- `sevilla_hidden_gems_top_by_neighborhood_v1.csv`

Si el ranking piloto es aceptable, pasaremos a crear el loader para cargar:

- catálogo local de platos Sevilla,
- menciones normalizadas con sentimiento,
- señales `place + dish`,
- candidatos Hidden Gems `sevilla_pilot`.
